# Week 2 - ML2 - Exercise 2: Simple RAG From PDFs

Build a tournament prep assistant for a fictional esports coaching staff.
It should answer strategy questions using the five team reports in `data`.
When the reports cannot answer a question, it should say so.

We will read the PDFs, split the text, retrieve useful passages, and ask a model
to answer using those passages. This is retrieval-augmented generation (RAG).

Work through one step at a time. Complete the five `TODO` tasks and write notes
where indicated. The other code is provided. Finish each TODO before running the
cells that depend on it.

## What you will practise

- Choose how to split documents into chunks.
- Inspect passages found with embeddings and FAISS.
- Write a prompt that keeps answers grounded in those passages.
- Test missing context and improve vague follow-up questions.

The packages are listed in `requirements.txt`. We use `pypdf` to read PDFs
and `langchain-text-splitters` only to split text.

## Before you start

Use Python 3.12. In a terminal in this repository, create a virtual environment:

```sh
python -m venv .venv
```

Activate it with `.venv\Scripts\Activate.ps1` in PowerShell, or
`source .venv/bin/activate` on macOS/Linux. Then install the packages:

```sh
python -m pip install -r requirements.txt
```

Copy `.env.example` to `.env` and add your OpenAI API key. Keep `.env` private.
The model names are supplied; you can change them in that file.

Start JupyterLab with `python -m jupyterlab`, select the `.venv` Python kernel,
and run this notebook from top to bottom. The five example PDFs are in `data`.

Embedding the chunks and running the examples makes API requests and may incur charges.
Re-running the embedding cell sends all chunks to the API again.


In [1]:
import os
from collections import Counter
from pathlib import Path

import faiss
import numpy as np
from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv(".env")

True

In [2]:
DATA_DIR = Path("data")
PDF_FILES = sorted(DATA_DIR.glob("*.pdf"))

if not PDF_FILES:
    raise FileNotFoundError("No PDFs found in data. Add PDF files first.")

PDF_FILES

[PosixPath('data/esports_overview_01_phantom_signal_valorant.pdf'),
 PosixPath('data/esports_overview_02_iron_tide_cs2.pdf'),
 PosixPath('data/esports_overview_03_neon_assembly_valorant.pdf'),
 PosixPath('data/esports_overview_04_vector_praetorians_cs2.pdf'),
 PosixPath('data/esports_overview_05_midnight_orbit_valorant.pdf')]

## Step 1 - Read PDFs

In [3]:
def read_pdf_text(pdf_path: Path) -> str:
    reader = PdfReader(str(pdf_path))
    pages = []
    for page in reader.pages:
        pages.append(page.extract_text() or "")
    return "\n".join(pages).strip()


documents = []
for path in PDF_FILES:
    text = read_pdf_text(path)
    documents.append(
        {
            "source": path.name,
            "text": text,
            "char_count": len(text),
        }
    )

documents[:2]

incorrect startxref pointer(1)
parsing for Object Streams
incorrect startxref pointer(1)
parsing for Object Streams
incorrect startxref pointer(3)
parsing for Object Streams
incorrect startxref pointer(1)
parsing for Object Streams
incorrect startxref pointer(1)
parsing for Object Streams


[{'source': 'esports_overview_01_phantom_signal_valorant.pdf',
  'text': 'Phantom Signal (VALORANT): A comeback built on emotional control and clearer calling\nThis overview describes a fictional esports team created for teaching purposes.\nPhantom Signal started the split looking like a team assembled from highlight clips rather than\nshared habits. Their early matches were loud and frantic, with heroic moments that disguised\nweak structure. Round reviews showed the same pattern every series: good opening reads, poor\nmid-round adaptation, and visible emotional spikes after lost advantage rounds. The coaches did\nnot panic about tactics first. They treated communication discipline as the first lever, arguing\nthat no playbook survives if players stop hearing each other when pressure rises.\nThe in-game leader, Riven, had inherited an identity that was too scripted for the roster she\nactually had. In previous teams she won with set executions and tightly sequenced utility, but\nthis 

## Step 2 - Choose the chunks (TODO 1)

A chunk is a short passage from a document. The splitter measures its size in characters.
An overlap repeats some characters in the next chunk so a sentence is less likely to lose context.

Choose `CHUNK_SIZE` and `CHUNK_OVERLAP` below. The overlap must be smaller than the chunk size.
Add a short comment explaining your choice.

Then run the supplied splitter and read the chunk counts. Look at a few chunks:
do they contain enough context to make sense?

In [8]:
# TODO 1: replace None with your chosen numbers of characters.
CHUNK_SIZE = 800
CHUNK_OVERLAP = 150

# 800 Zeichen ≈ ein Absatz: genug Kontext (Team + Massnahme), aber noch ein klares Thema pro Chunk.
# 150 Overlap (~20 %), damit Sätze an der Grenze nicht auseinandergerissen werden.

In [9]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)

chunk_records = []
for doc in documents:
    chunks = splitter.split_text(doc["text"])
    for i, chunk in enumerate(chunks):
        chunk_records.append(
            {
                "source": doc["source"],
                "chunk_id": f"{doc['source']}_chunk_{i}",
                "chunk_text": chunk,
            }
        )

chunk_records[:10]

[{'source': 'esports_overview_01_phantom_signal_valorant.pdf',
  'chunk_id': 'esports_overview_01_phantom_signal_valorant.pdf_chunk_0',
  'chunk_text': 'Phantom Signal (VALORANT): A comeback built on emotional control and clearer calling\nThis overview describes a fictional esports team created for teaching purposes.\nPhantom Signal started the split looking like a team assembled from highlight clips rather than\nshared habits. Their early matches were loud and frantic, with heroic moments that disguised\nweak structure. Round reviews showed the same pattern every series: good opening reads, poor\nmid-round adaptation, and visible emotional spikes after lost advantage rounds. The coaches did\nnot panic about tactics first. They treated communication discipline as the first lever, arguing\nthat no playbook survives if players stop hearing each other when pressure rises.'},
 {'source': 'esports_overview_01_phantom_signal_valorant.pdf',
  'chunk_id': 'esports_overview_01_phantom_signal_va

In [10]:
chunk_count_by_source = Counter(item["source"] for item in chunk_records)

print("Number of documents:", len(documents))
print("Number of chunks:", len(chunk_records))
print("\nChunks per source:")
for source, count in chunk_count_by_source.items():
    print(f"- {source}: {count}")

Number of documents: 5
Number of chunks: 36

Chunks per source:
- esports_overview_01_phantom_signal_valorant.pdf: 7
- esports_overview_02_iron_tide_cs2.pdf: 8
- esports_overview_03_neon_assembly_valorant.pdf: 7
- esports_overview_04_vector_praetorians_cs2.pdf: 7
- esports_overview_05_midnight_orbit_valorant.pdf: 7


## Step 3 - Build FAISS Retriever


### 3.1 Embed all chunks once

An embedding represents a passage as a list of numbers. We create one for each chunk.
The query will use the same embedding model so its vector can be compared with them.

Run this cell once. If you change the embedding model or the chunks, run it again
and rebuild the index in the next section.

In [11]:
embedding_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "text-embedding-3-small")


def embed_texts(texts, batch_size=64):
    vectors = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        response = embedding_client.embeddings.create(
            model=EMBEDDING_MODEL,
            input=batch,
        )
        vectors.extend(item.embedding for item in response.data)
    return np.array(vectors, dtype="float32")


chunk_texts = [item["chunk_text"] for item in chunk_records]
chunk_embeddings = embed_texts(chunk_texts)
faiss.normalize_L2(chunk_embeddings)
print("Chunk embeddings shape:", chunk_embeddings.shape)

Chunk embeddings shape: (36, 1536)


### 3.2 Build the vector index

FAISS stores vectors and lets us search nearest neighbors efficiently.
Here we use inner product on normalized vectors, which behaves like cosine similarity.


In [12]:
faiss_index = faiss.IndexFlatIP(chunk_embeddings.shape[1])
faiss_index.add(chunk_embeddings)
print("Vectors stored in FAISS:", faiss_index.ntotal)

Vectors stored in FAISS: 36


### 3.3 Retrieve passages (TODO 2)

The function below embeds the question, searches FAISS, and returns matching chunks.
`top_k` is the number of matches to return.

Run the function cell first. Then run the two example questions with `TOP_K = 3`.
Read the three passages returned for each question. A higher similarity score means
a closer match, but you still need to judge whether the text answers the question.

Add a short observation in the marked comment at the end of the example cell.

In [13]:
def retrieve_chunks(query: str, top_k: int = 3):
    query_response = embedding_client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=query,
    )
    query_vector = np.array([query_response.data[0].embedding], dtype="float32")
    faiss.normalize_L2(query_vector)

    k = min(top_k, len(chunk_texts))
    scores, indices = faiss_index.search(query_vector, k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        item = chunk_records[int(idx)]
        results.append(
            {
                "source": item["source"],
                "chunk_id": item["chunk_id"],
                "similarity": float(score),
                "chunk_text": item["chunk_text"],
            }
        )

    return results

In [14]:
# TODO 2: run the two retrieval examples, then write your observation below.
# Start with three retrieved chunks per question.
TOP_K = 3

q1 = "How did Phantom Signal improve after early losses?"
q2 = "What role did communication protocols play in Neon Assembly's turnaround?"

for query in [q1, q2]:
    print("=" * 80)
    print("Query:", query)
    results = retrieve_chunks(query, top_k=TOP_K)
    for r in results:
        print(f"- {r['chunk_id']} | similarity={r['similarity']:.4f}")
        print(f"  {r['chunk_text'][:180]}...")

# Observation: do the passages discuss the right team and answer the question?
# ...

Query: How did Phantom Signal improve after early losses?
- esports_overview_01_phantom_signal_valorant.pdf_chunk_2 | similarity=0.6529
  was available at second forty-five."
A major turning point came after a painful loss where Phantom Signal led by six rounds and then
collapsed. Rather than focusing on mechanics, s...
- esports_overview_01_phantom_signal_valorant.pdf_chunk_0 | similarity=0.6197
  Phantom Signal (VALORANT): A comeback built on emotional control and clearer calling
This overview describes a fictional esports team created for teaching purposes.
Phantom Signal ...
- esports_overview_01_phantom_signal_valorant.pdf_chunk_5 | similarity=0.6126
  reduced emotional contamination before training. The team reported lower tilt carryover and more
focused review sessions. Their head coach described the goal as building a room whe...
Query: What role did communication protocols play in Neon Assembly's turnaround?
- esports_overview_03_neon_assembly_valorant.pdf_chunk_5 | similarity

## Step 4 - Write the prompt (TODO 3)

Replace the instruction line in `PROMPT_TEMPLATE` below. Tell the model to:

- Act as a tournament prep assistant and give concise, practical advice for coaches.
- Focus on strategy, such as communication, leadership and preparation.
- Use only the supplied context.
- Say exactly `I don't know based on the provided context.` when that context is insufficient.

Keep `{context}` and `{question}` in the template. The supplied generation function
will fill them with retrieved passages and the user's question.

In [15]:
# TODO 3: write your instructions above Context.
PROMPT_TEMPLATE = """
You are a Tournament Preparation Assistant for Coaches. Your role is to support coaches in strategically preparing for tournaments, with a focus on communication, leadership, and preparation.

Always provide short, clear, and practical responses. Focus on concrete, actionable recommendations that coaches can apply immediately.

Base your answers exclusively on the provided context. Do not use external knowledge, make assumptions, or introduce information that is not explicitly supported by the context.

If the provided context is insufficient to answer a question reliably, respond with exactly:

I don't know based on the provided context.

Do not add any further explanations or additional information in this case.

Context:
{context}

Question:
{question}
"""

## Step 5 - Retrieve and answer

The next function joins the retrieved chunks into the prompt and sends it to OpenAI.
It returns both the answer and the passages it used.

Run the function definition, then the Iron Tide example. Read the answer alongside
the retrieved passages. Can you find support for its claims?

In [16]:
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
MODEL_NAME = os.getenv("OPENAI_MODEL", "gpt-6-luna")


def generate_answer(question: str, top_k: int = 3):
    retrieved = retrieve_chunks(question, top_k=top_k)
    context = "\n\n".join(item["chunk_text"] for item in retrieved)
    prompt = PROMPT_TEMPLATE.format(context=context, question=question)

    response = client.responses.create(
        model=MODEL_NAME,
        input=prompt,
    )
    answer = response.output_text
    return answer, retrieved

In [17]:
test_question = "What strategic changes helped Iron Tide recover from bad starts?"
answer, used_chunks = generate_answer(test_question, top_k=TOP_K)

print("Question:", test_question)
print("\nAnswer:\n", answer)
print("\nRetrieved Chunks:")
for item in used_chunks:
    print(f"- {item['chunk_id']} | similarity={item['similarity']:.4f}")

Question: What strategic changes helped Iron Tide recover from bad starts?

Answer:
 They stopped spiraling after poor starts: communication stayed functional, and mid-round calls followed available information instead of emotional urgency. Their flexible defaults and disciplined transitions also helped them regain control.

Retrieved Chunks:
- esports_overview_02_iron_tide_cs2.pdf_chunk_5 | similarity=0.6847
- esports_overview_02_iron_tide_cs2.pdf_chunk_6 | similarity=0.6745
- esports_overview_02_iron_tide_cs2.pdf_chunk_7 | similarity=0.6623


## Step 6 - Test failure cases (TODO 4)

Run the two questions below, one vague and one unrelated to the reports.
Read the retrieved passages as well as the answers.

Add a short note explaining what happened in each case. Did the model acknowledge
missing information? Did the passages actually help answer the question?

In [18]:
vague_query = "Tell me more."
out_of_scope_query = "How can I optimize my home Wi-Fi router placement?"

for q in [vague_query, out_of_scope_query]:
    ans, ret = generate_answer(q, top_k=TOP_K)
    print("=" * 80)
    print("Question:", q)
    print("Answer:", ans)
    print("Top retrieved chunk IDs:", [item["chunk_id"] for item in ret])


# TODO 4: explain what happened and why.
# Vague question: ...
# Unrelated question: ...

Question: Tell me more.
Answer: Phantom Signal improved by fixing the conditions that let tactics work:

- **Stabilized communication first.** Early on, strong opening reads often broke down into poor mid-round adaptation. The team prioritized hearing and responding to one another under pressure.
- **Made calls more precise.** Riven shifted from longer, theatrical speeches to short triggers tied to information thresholds.
- **Managed emotional carryover.** Reducing tilt before training helped players focus on the next decision and made review sessions more productive.
- **Built trust alongside role comfort.** The team treated these as connected, especially for high-variance duelists.
- **Added depth after reliability improved.** Their stronger maps showed proactive defensive calls, faster trades at contested choke points, and steadier tempo changes when opponents adapted.

The practical lesson: improve the communication and emotional habits that support tactics before adding complexity

## Step 7 - Improve vague follow-ups (TODO 5)

A question such as `Tell me more.` does not name a topic. We can add the previous
question to give the retriever something more specific to search for.

Complete `rewrite_query_if_vague` below:

1. Convert the question to lowercase and remove surrounding spaces and final punctuation.
   The methods `.lower()`, `.strip()` and `.rstrip(".?!")` may help.
2. If it matches a phrase in `vague_patterns`, return
   `f"Provide more details about: {last_user_query}"`.
3. Otherwise, return the original question unchanged.

The helper below it is provided. It uses your rewritten question with the existing
`generate_answer` function. The previous question is passed in explicitly; it comes
from the test cases below.

In [21]:
# TODO 5: implement this function.
def rewrite_query_if_vague(query: str, last_user_query: str):
    vague_patterns = ["tell me more", "more", "what about that", "and this"]
    cleaned = query.lower().strip().rstrip(".?!")
    if cleaned in vague_patterns:
        return f"Provide more details about: {last_user_query}"
    return query


# Provided: use the rewritten question to retrieve passages and generate an answer.
def generate_answer_with_rewrite(question: str, last_user_query: str, top_k: int = 3):
    rewritten = rewrite_query_if_vague(question, last_user_query)
    answer, retrieved = generate_answer(rewritten, top_k=top_k)
    return rewritten, answer, retrieved

In [22]:
test_cases = [
    {
        "question": "Tell me more.",
        "last_user_query": "How did Midnight Orbit stabilize communication after coaching changes?",
    },
    {
        "question": "And this?",
        "last_user_query": "Why did Vector Praetorians simplify their strategy book?",
    },
]

records = []
for case in test_cases:
    q = case["question"]
    last_q = case["last_user_query"]

    base_answer, _ = generate_answer(q, top_k=TOP_K)
    rewritten_q, improved_answer, _ = generate_answer_with_rewrite(q, last_user_query=last_q, top_k=TOP_K)

    records.append(
        {
            "original_query": q,
            "rewritten_query": rewritten_q,
            "baseline_answer_preview": base_answer[:180],
            "improved_answer_preview": improved_answer[:180],
        }
    )

for row in records:
    print("=" * 80)
    print("Original query:", row["original_query"])
    print("Rewritten query:", row["rewritten_query"])
    print("Baseline preview:", row["baseline_answer_preview"])
    print("Improved preview:", row["improved_answer_preview"])


# TODO 5: compare the previews above.
# Did rewriting help? What limitation remains?
# ...

Original query: Tell me more.
Rewritten query: Provide more details about: How did Midnight Orbit stabilize communication after coaching changes?
Baseline preview: Phantom Signal’s turnaround came from fixing communication and emotional control before adding tactical complexity:

- **Stabilized communication:** Calls became more precise and t
Improved preview: After assistant responsibilities shifted, Midnight Orbit kept its review frameworks and team language consistent. They avoided reactionary tactical changes and focused on gradual, 
Original query: And this?
Rewritten query: Provide more details about: Why did Vector Praetorians simplify their strategy book?
Baseline preview: I don't know based on the provided context.
Improved preview: They simplified because the team had many setups but struggled to choose the right one under stress. In officials, they often lost momentum after early disruptions, and low-frequen


## Before you finish

- [ ] Explain your chunk size and overlap.
- [ ] Comment on retrieval for both example questions.
- [ ] Write and test your grounded prompt.
- [ ] Explain the two failure cases.
- [ ] Compare answers before and after query rewriting.
- [ ] Complete the reflection below.

Optional extension: refine your prompt and repeat a failure case.
The worked solution in `ml2_exercise2_solution.ipynb` demonstrates query rewriting.

## Final reflection

Answer briefly, using at most eight lines in total.

1. What worked best in your pipeline?
2. What was the biggest failure mode?
3. Which design choice mattered most?
4. What is one limitation when using incomplete or biased reports?

Your reflection: ...